In [ ]:
import io
import sys
from contextlib import redirect_stdout
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from shapely.geometry import Polygon
from shapely.geometry import Point
from datetime import datetime
from dateutil.relativedelta import relativedelta
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
import pingouin as pg
import csv

# Alistamiento de datos para correlaciones

Se corre cuando se tiene una nueva base de datos para alistar el archivo, sólo se debe hacer una vez, y dicho archivo queda guardado, mostrandose en el siguiente bloque

In [ ]:
# Get the path to the file one directory up
# Get current working directory and go up one level
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
file_path = os.path.join(parent_dir, 'common/land_cover/e_cobertura_tierra_2020_admin.shp')


# Load the file into a GeoDataFrame
gdf = gpd.read_file(file_path)
gdf.plot('nivel_1',legend=True)

In [ ]:
dfg.columns

In [ ]:
mydict1={'1':'1. Territorios artificializados','2':'2. Territorios agrícolas','3':'3. Bosques y áreas seminaturales','4':'4. Áreas húmedas','5':'5. Superficies de agua'}
mydict1

In [ ]:
Nivel3_path = os.path.join(parent_dir, 'common/Nivel3.csv')
N3_path = os.path.join(parent_dir, 'common/N3.csv')

with open(Nivel3_path, mode='r') as infile:
    reader = csv.reader(infile)
    with open(N3_path, mode='w') as outfile:
        writer = csv.writer(outfile)
        mydict = {rows[1]:rows[0] for rows in reader}
mydict

In [ ]:
dfg['Leyenda_1']=dfg['nivel_1'].map(mydict1)
dfg[['Leyenda_1','nivel_1']]

In [ ]:
dfg['Leyenda_3']=dfg['nivel_3'].map(mydict)
dfg[['Leyenda_3','nivel_3']]

In [ ]:
encoded_df = pd.get_dummies(dfg, columns=["Leyenda_3"])  # Encode the "leyenda 3" column
encoded_df1 = pd.get_dummies(dfg, columns=["Leyenda_1"])  # Encode the "Leyenda 1" column
#print(encoded_df1.columns)
encoded_df = pd.concat([encoded_df,encoded_df1[['Leyenda_1_1. Territorios artificializados', 'Leyenda_1_2. Territorios agrícolas', 'Leyenda_1_3. Bosques y áreas seminaturales','Leyenda_1_4. Áreas húmedas',
       'Leyenda_1_5. Superficies de agua']]], axis=1)
encoded_df['groupb']=0
encoded_df['geometry']

# Suplemental
Se debe correr la sección alistamiento de datos hastala línea donde se carga df_s y se muestra un mapa

In [ ]:
encoded_df45 = encoded_df[(encoded_df['Leyenda_1_4. Áreas húmedas']>0)+(encoded_df['Leyenda_1_5. Superficies de agua']>0)]#tabla para eliminar el glint en la plataforma continental

In [ ]:
encoded_df45

In [ ]:

satDataPath = os.path.join(parent_dir, 'common/colombia_result_corrected.csv')

df_s = pd.read_csv(satDataPath)

df_s

In [ ]:
df_s['coordinate_x'] = df_s['longitude'].apply(lambda x: [x])
df_s['coordinate_y'] = df_s['latitude'].apply(lambda x: [x])
df_s['coordinates'] = df_s['coordinate_x'] + df_s['coordinate_y']

delta_y = 0.01/2
delta_x = 0.01/2

df_s['geometry'] = df_s['coordinates'].apply(
    lambda x: Polygon([
        (x[0] - delta_x, x[1] - delta_y),
        (x[0] - delta_x, x[1] + delta_y),
        (x[0] + delta_x, x[1] + delta_y),
        (x[0] + delta_x, x[1] - delta_y)
    ]))

df_s

In [ ]:
l_in = list(encoded_df45.columns)

l_in



In [ ]:
while not l_in[0]=='Leyenda_3_1.1.1. Tejido urbano continuo':
    l_in.pop(0)


l_in.remove('groupb')

l_in

In [ ]:
for i in l_in:
    encoded_df45.loc[:, i] = encoded_df[i] * encoded_df['SHAPE_Area']

encoded_df45

In [ ]:
# for j in df_s.index:
#     #print(j)
#     f=encoded_df45[df_s['geometry'][j].intersects(encoded_df45['geometry'])]
#     f1=f.groupby('groupb')[l_in].aggregate('sum') # Se suma el área total intersectada
#     if not len(f1)==0:
#       for i in l_in:
#         df_s.loc[j, i]=f1[i].iloc[0].copy()
#     if j%5e4==0:
#       print(j)
# listind=[]
# for j in encoded_df45.index:
#     f = df_s[encoded_df45['geometry'][j].intersects(df_s['geometry'])]
#     #f1=f.groupby('groupb')[l_in].aggregate('sum') # Se suma el área total intersectada
#     listind.append(f.index)
# df_sfil=df_s[~listind]
# df_sfil.to_csv('colombia_results_filtrado.csv')


# #correr hasta acá


listind = []

for j in encoded_df45.index:
    f = df_s[encoded_df45['geometry'][j].intersects(df_s['geometry'])]
    listind.extend(f.index.tolist())  # Use extend instead of append to flatten the list


In [ ]:

# Convert to unique indices
listind = list(set(listind))  # Remove duplicates if any

# Filter the dataframe
df_sfil = df_s[~df_s.index.isin(listind)]  # Use isin() for filtering
df_sfil.to_csv('colombia_results_filtrado.csv')

----------------------------------------------------